In [37]:
import os
import glob
import pandas as pd
import base64
from jinja2 import Template
import weasyprint

def file_to_data_uri(filepath):
    if not filepath or not os.path.exists(filepath):
        return None
    ext = os.path.splitext(filepath)[1].lower().replace('.', '')
    if ext == 'jpg':
        ext = 'jpeg'
    with open(filepath, 'rb') as f:
        data = base64.b64encode(f.read()).decode('utf-8')
    return f"data:image/{ext};base64,{data}"

def generar_catalogo(excel_path, carpeta_imagenes='.', archivo_salida_pdf='Catalogo_SMG_Mayorista.pdf', solo_primeras_hojas=True):
    # 1. Leer el Excel preservando exactamente el orden original de las filas (sin reordenar ni ordenar alfabéticamente)
    df = pd.read_excel(excel_path, sheet_name='PRECIO_MAY')
    df.columns = df.columns.str.strip().str.upper()
    
    col_cod = [c for c in df.columns if 'COD' in c or 'ART' in c or 'PROD' in c][0]
    col_precio = [c for c in df.columns if 'PRECIO' in c or 'VALOR' in c][0]
    
    # 2. Mapear imágenes locales para consulta rápida por código
    extensiones = ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.PNG', '*.JPG', '*.JPEG')
    archivos_img = []
    
    for root, _, _ in os.walk(carpeta_imagenes):
        for ext in extensiones:
            archivos_img.extend(glob.glob(os.path.join(root, ext)))
    
    mapa_imagenes = {}
    for ruta in archivos_img:
        nombre_sin_ext = os.path.splitext(os.path.basename(ruta))[0].strip().upper()
        mapa_imagenes[nombre_sin_ext] = file_to_data_uri(ruta)
    
    # 3. Construir la lista de productos SIGUIENDO ESTRICTAMENTE EL ORDEN DE FILAS DEL EXCEL
    productos = []
    for _, fila in df.iterrows():
        cod_raw = str(fila[col_cod]).strip()
        cod_clean = cod_raw.upper()
        
        precio_val = fila[col_precio]
        try:
            precio_fmt = f"${float(precio_val):,.0f}".replace(",", ".")
        except:
            precio_fmt = f"${precio_val}"
            
        img_b64 = mapa_imagenes.get(cod_clean, None)
        
        productos.append({
            'cod': cod_raw,
            'precio': precio_fmt,
            'imagen': img_b64
        })
    
    # 4. Agrupar de a 6 productos por página manteniendo el orden
    paginas = [productos[i:i + 6] for i in range(0, len(productos), 6)]
    
#    if solo_primeras_hojas:
#        paginas = paginas[:2]
    
    portada_b64 = file_to_data_uri('portada.png')
    fondo_b64 = file_to_data_uri('fondo.png')
    logo_b64 = file_to_data_uri('logo.png')
    
    html_template = """
    <!DOCTYPE html>
    <html lang="es">
    <head>
    <meta charset="UTF-8">
    <style>
      @page {
        size: 210mm 297mm;
        margin: 0;
      }
      html, body {
        margin: 0;
        padding: 0;
        width: 210mm;
        height: 297mm;
        font-family: 'Times New Roman', serif;
        color: #333;
      }
      * {
        box-sizing: border-box;
      }
      
      .cover-page {
        width: 210mm;
        height: 297mm;
        background-image: url('{{ portada_b64 }}');
        background-size: 100% 100%;
        background-position: center;
        background-repeat: no-repeat;
        page-break-after: always;
      }

      .catalog-page {
        width: 210mm;
        height: 297mm;
        padding: 0;
        margin: 0;
        position: relative;
        background-image: url('{{ fondo_b64 }}');
        background-size: 100% 100%;
        background-position: center;
        background-repeat: no-repeat;
        page-break-after: always;
        overflow: hidden;
      }
      
      .header {
        position: absolute;
        top: 2mm;
        left: 0;
        width: 210mm;
        height: 48mm;
        text-align: center;
        display: flex;
        align-items: center;
        justify-content: center;
        z-index: 10;
      }
      .logo-img {
        max-height: 40mm;
        max-width: 160mm;
        object-fit: contain;
        display: block;
        margin: 0 auto;
        transform: scale(2.6);
        transform-origin: center center;
      }
      
      .divider-line {
        position: absolute;
        top: 52mm;
        height: 232mm;
        left: 105mm;
        width: 1.5px;
        background-color: #c9a050;
        opacity: 0.85;
        z-index: 1;
      }
      
      .grid-container {
        position: absolute;
        top: 52mm;
        left: 0;
        width: 210mm;
        height: 234mm;
      }

      .item-row {
        height: 78mm;
        width: 100%;
        clear: both;
      }

      .item-col {
        float: left;
        width: 105mm;
        height: 78mm;
        text-align: center;
        padding: 0 4mm;
      }

      .img-wrapper {
        height: 48mm;
        width: 100%;
        display: flex;
        align-items: center;
        justify-content: center;
        overflow: hidden;
      }

      .product-img {
        max-height: 44mm;
        max-width: 90mm;
        object-fit: contain;
        display: block;
        transform: scale(3.2);
      }

      .img-missing {
        height: 42mm;
        width: 70mm;
        border: 1px dashed #c9a050;
        border-radius: 6px;
        line-height: 42mm;
        color: #a88232;
        font-size: 10pt;
        margin: 0 auto;
      }

      .text-box {
        margin-top: 2mm;
        width: 100%;
        clear: both;
      }

      .cod {
        font-size: 15pt;
        font-weight: bold;
        color: #b38b22;
        letter-spacing: 0.5px;
        line-height: 1.25;
        display: block;
        margin: 0;
      }

      .precio {
        font-size: 15pt;
        font-weight: bold;
        color: #b38b22;
        line-height: 1.25;
        display: block;
        margin: 0;
      }
    </style>
    </head>
    <body>

      {% if portada_b64 %}
      <div class="cover-page"></div>
      {% endif %}

      {% for pagina in paginas %}
      <div class="catalog-page">
        
        <div class="header">
          {% if logo_b64 %}
            <img src="{{ logo_b64 }}" class="logo-img" />
          {% endif %}
        </div>

        <div class="divider-line"></div>

        <div class="grid-container">
          {% for i in range(0, pagina|length, 2) %}
          <div class="item-row">
            
            <div class="item-col">
              <div class="img-wrapper">
                {% if pagina[i].imagen %}
                  <img src="{{ pagina[i].imagen }}" class="product-img" />
                {% else %}
                  <div class="img-missing">Sin imagen</div>
                {% endif %}
              </div>
              <div class="text-box">
                <div class="cod">{{ pagina[i].cod }}</div>
                <div class="precio">{{ pagina[i].precio }}</div>
              </div>
            </div>

            <div class="item-col">
              {% if i + 1 < pagina|length %}
                <div class="img-wrapper">
                  {% if pagina[i+1].imagen %}
                    <img src="{{ pagina[i+1].imagen }}" class="product-img" />
                  {% else %}
                    <div class="img-missing">Sin imagen</div>
                  {% endif %}
                </div>
                <div class="text-box">
                  <div class="cod">{{ pagina[i+1].cod }}</div>
                  <div class="precio">{{ pagina[i+1].precio }}</div>
                </div>
              {% endif %}
            </div>

          </div>
          {% endfor %}
        </div>

      </div>
      {% endfor %}

    </body>
    </html>
    """
    
    template = Template(html_template)
    html_renderizado = template.render(
        paginas=paginas,
        portada_b64=portada_b64,
        fondo_b64=fondo_b64,
        logo_b64=logo_b64
    )
    
    with open('temp_catalogo.html', 'w', encoding='utf-8') as f:
        f.write(html_renderizado)
        
    weasyprint.HTML('temp_catalogo.html').write_pdf(archivo_salida_pdf)
    print("✅ Catálogo generado respetando el orden exacto del Excel.")

if __name__ == '__main__':
    generar_catalogo('precios.xlsx', '.')

✅ Catálogo generado respetando el orden exacto del Excel.
